### Ingestion
Here we will follow below mnetioned steps:

1. Read the mapping sheet and create the table w.r.t it

2. add the all the SCD2 logic on each table
    - MERGE command based
    - APPLY SCD
    - APPLY Changes
    - using the haskey based logic

3. add the audit column and config column:

    Audit column:

    -     add_date
    -     delete_date
    -     last_modified_date
    -     is_active
    -     is_incremental
    -     hashkey

    Config column:
    -     data_source_id
    -     data_source
    -     dataset
    -     data_version

    


In [0]:
%run /Users/pioneerspd@gmail.com/Metadatadriven_proj/HA_control

### HA_control
Reads the weekly load sheet (`Weekly_load_stats.csv`), checks it, and tells a layer notebook which tables to run.

Load it with `%run ./HA_control` as the only line of its own cell. It defines functions only, so it is safe to `%run`.


In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp
import logging
import pandas as pd
import json,csv
import ast
from datetime import datetime
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force =True
)

logger = logging.getLogger(__name__)

In [0]:
import os

TYPE_MAP = {"varchar": "string", "string": "string", "int": "int",
            "bigint": "bigint", "double": "double", "date": "date",
            "timestamp": "timestamp", "boolean": "boolean"}

# one timestamp per run: every row a fact load writes gets the same add_date, so "the rows of the
# latest run" is a plain filter and old rows never have to be rewritten
RUN_TS = datetime.now()
CKPT_BASE = "/Volumes/dab/healthcare_datasets/healthcare_vol/data/checkpoint"
VOL_DATA_BASE = "/Volumes/dab/healthcare_datasets/healthcare_vol/data"

def get_vol_path(layer, short_name):
    """Volume path for a layer+table, e.g. ingestion/ingestion_mx_claim.
    Creates the parent directory so Delta writes don't fail on a missing path."""
    path = f"{VOL_DATA_BASE}/{layer}/{short_name}"
    os.makedirs(path, exist_ok=True)
    return path

def get_data_version(cfg):
    "data_version is set per layer in dev_config.json; fail loudly instead of guessing a value"
    v = str(cfg.get("data_version") or "").strip()
    if not v:
        raise ValueError("dev_config.json has no data_version for this layer")
    return v

def load_config(path):
    """Load JSON config, auto-repairing missing commas between key-value pairs.
    The config file in the shared volume is occasionally re-saved without commas;
    this catches the breakage, repairs it in place, and re-validates."""
    with open(path, 'r') as f:
        raw = f.read()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        logger.warning(f"{path}: JSON parse error ({e}), attempting auto-repair")
        lines = raw.split('\n')
        fixed_lines = []
        for i, line in enumerate(lines):
            stripped = line.rstrip()
            if (stripped.endswith('"') and not stripped.endswith(',')
                    and not stripped.endswith('{') and not stripped.endswith('[')
                    and not stripped.endswith(':')
                    and i + 1 < len(lines)
                    and lines[i + 1].lstrip().startswith('"')):
                line = stripped + ','
            fixed_lines.append(line)
        fixed = '\n'.join(fixed_lines)
        with open(path, 'w') as f:
            f.write(fixed)
        logger.info(f"{path}: auto-repaired missing commas, re-validating")
        with open(path, 'r') as f:
            return json.load(f)

In [0]:
def read_mapping(map_path):
    # add .option("sep", "\t") if the sheet is tab-separated
    return spark.read.option("header", True).csv(map_path)

In [0]:
def parse_inc_arg(text):
    if not text or not text.strip():
        return {}
    return ast.literal_eval(text.strip())

In [0]:
def build_ingestion_df(map_df, cdm_table, src_catalog, src_schema):
    rules = (map_df
             .filter(col("is_active_map").cast("int") == 1)
             .filter(lower(trim("cdm_table")) == cdm_table.lower())
             .collect())
    if not rules:
        raise ValueError(f"No active mapping rows for {cdm_table!r}")

    src_table = rules[0]["src_table"].strip()
    # `rules` is a list of Rows, not a DataFrame
    target_types = sorted({r["target_type"].strip().lower()
        for r in rules
        if r["target_type"] and r["target_type"].strip()})
    # mapping holds the short source name; staging tables are named staging_<...>
    # prefer the staging tables: in the consolidated dev schema the bare src_table
    # (e.g. dev.provider) also exists as the moved dim table and would shadow the
    # real staging source, so it is only a last-resort fallback
    src_candidates = [f"staging_{src_table}", f"staging_{cdm_table}", src_table]
    src_table_name = next(
        (f"{src_catalog}.{src_schema}.{c}" for c in src_candidates
         if spark.catalog.tableExists(f"{src_catalog}.{src_schema}.{c}")),
        f"{src_catalog}.{src_schema}.{src_table}",
    )

    spark.sql(f"""
    ALTER TABLE {src_table_name}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true);          
    """)
    if "fact" in target_types:
        src_df = (spark.readStream
            .option("readChangeFeed", "true")
            .table(src_table_name)
            .filter(col("_change_type") != "update_preimage"))
    else:
        src_df = spark.read.table(src_table_name)
    
    src_name = (rules[0]["src_name"] or "").strip()
    value_defaults = {
        "data_source_id": src_name.upper(),
        "data_source": src_name,
        "dataset": src_table,
    }

    exprs, key_cols, non_key_cols = [], [], []
    for r in rules:
        tgt      = r["cdm_column"].strip()
        src_col  = (r["src_column"] or "").strip()
        sql_tx   = (r["sql_tx"] or "").strip()
        dtype    = TYPE_MAP.get((r["data_type"] or "string").strip().lower(), "string")
        if dtype == "int":
            # widen to bigint: provider_id holds 10-digit values that overflow INT
            dtype = "bigint"

        if sql_tx:
            col_expr = expr(sql_tx)
        elif src_col:
            col_expr = col(f"`{src_col}`")
        elif tgt in value_defaults:
            col_expr = lit(value_defaults[tgt])
        else:
            raise ValueError(f"{cdm_table}.{tgt}: no src_column, sql_tx or default value")

        if dtype == "date":
            # staging dates arrive as MM/dd/yyyy strings, not ISO yyyy-MM-dd
            exprs.append(to_date(col_expr, "MM/dd/yyyy").alias(tgt))
        else:
            exprs.append(col_expr.cast(dtype).alias(tgt))
        (key_cols if (r["map_type"] or "").strip().upper() == "PRIMARY" else non_key_cols).append(tgt)

    df = src_df.select(*exprs)

    # inc_arg appears only on the first row of each table in your sheet
    inc = {}
    for r in rules:
        if not r["inc_arg"]:
            continue
        try:
            inc = parse_inc_arg(r["inc_arg"])
        except (ValueError, SyntaxError):
            inc = {}
        if isinstance(inc, str):
            # Spark's CSV reader doubles the quotes around values, so the dict
            # comes back as a string with unquoted values: {'hash_key_reqd':Y}
            try:
                inc = ast.literal_eval(inc.replace(":Y}", ":'Y'}"))
            except (ValueError, SyntaxError):
                inc = {}
        if isinstance(inc, dict) and inc:
            break
    opts = (inc.get("EXCEPT_KEY_COLUMN_CHECKSUM_LOAD")
        or inc.get("ALL_COLUMN_CHECKSUM_LOAD")
        or {})
    if opts.get("hash_key_reqd") == "Y" and non_key_cols:
        df = df.withColumn("row_hash", sha2(concat_ws("||",
             *[coalesce(col(c).cast("string"), lit("<null>")) for c in non_key_cols]), 256))

    scd = (rules[0]["scd"] or "").strip().upper() == "Y"
    return df, key_cols, scd,target_types

In [0]:
def SCD_UPSERT_logic(df,key,target,layer="ingestion"):
    "here we will apply the SCD with the upsert logic"
    view = f"ing_{target.replace('.', '_')}"
    df.createOrReplaceTempView(view)

    # columns with special handling in the MERGE
    audit_cols = {"hashkey", "row_hash", "add_date", "delete_date",
                  "last_modified_date", "is_active", "is_incremental"}

    # keys arrive as a list from build_ingestion_df; accept a single name too
    keys = list(key) if isinstance(key, (list, tuple)) else [key]

    # everything else in the dataframe is copied as-is
    value_cols = [c for c in df.columns if c not in keys and c not in audit_cols]

    if "row_hash" not in df.columns:
        raise ValueError(f"{target}: source dataframe has no row_hash column")

    set_vals = ",\n".join(f"{c} = src.{c}" for c in value_cols)
    ins_cols = ", ".join([*keys, *value_cols])
    ins_vals = ", ".join([f"src.{k}" for k in keys] + [f"src.{c}" for c in value_cols])
    on_expr  = " and ".join(f"tgt.{k} = src.{k}" for k in keys)

    # first run: create the target table with the audit columns
    if not spark.catalog.tableExists(target):
        short_name = target.split('.')[-1]
        vol_path = get_vol_path(layer, short_name)
        init_df = (df
         .withColumnRenamed("row_hash", "hashkey")
         .withColumn("add_date", current_timestamp())
         .withColumn("delete_date", to_timestamp(lit("2099-12-31 00:00:00")))
         .withColumn("last_modified_date", current_timestamp())
         .withColumn("is_active", lit(1))
         .withColumn("is_incremental", lit(1)))
        init_df.write.format("delta").mode("overwrite").saveAsTable(target)
        init_df.write.format("delta").mode("overwrite").save(vol_path)
        logger.info(f"{target}: initial full load (table + volume: {vol_path})")
        return None

    # audit-column semantics:
    #   not matched (new row)              -> fresh audit columns, is_incremental = 1
    #   matched, hash differs (changed row) -> hashkey, add_date, delete_date, last_modified_date
    #                                         all refreshed; is_active = 1; is_incremental = 1
    #   matched, hash identical (unchanged)  -> only is_active = 1 and is_incremental = 0;
    #                                         add_date/delete_date/last_modified_date untouched
    dim_df = spark.sql(f"""
    MERGE INTO {target} as tgt
    using {view} as src
    on {on_expr}
    when matched and tgt.hashkey <> src.row_hash
    then update set 
    {set_vals},
    tgt.hashkey = src.row_hash,
    tgt.add_date = current_timestamp(),
    tgt.delete_date = '2099-12-31 00:00:00',
    tgt.last_modified_date = current_timestamp(),
    tgt.is_active = 1,
    tgt.is_incremental = 1
    when matched and tgt.hashkey = src.row_hash
    then update set 
    tgt.is_active = 1,
    tgt.is_incremental = 0,
    tgt.data_version = src.data_version
    when not matched
    then insert ({ins_cols}, 
    hashkey,
    add_date, 
    delete_date, 
    last_modified_date, 
    is_active, 
    is_incremental)
    VALUES ({ins_vals}, 
    src.row_hash,
    current_timestamp(), 
    '2099-12-31 00:00:00', 
    current_timestamp(), 
    1, 
    1)""")
    # sync volume copy with the current table state after MERGE
    short_name = target.split('.')[-1]
    vol_path = get_vol_path(layer, short_name)
    spark.table(target).write.format("delta").mode("overwrite").save(vol_path)
    logger.info(f"{target}: MERGE done, volume synced to {vol_path}")
    return dim_df


In [0]:
import os

TEMPLATE_PATH = "/Volumes/dab/healthcare_datasets/healthcare_vol/sql/SCD2_fact.sql"
OUT_DIR       = "/Workspace/Users/pioneerspd@gmail.com/Metadatadriven_proj/pipeline/"      # where the pipeline reads its source files

def scd_cdc_logic(df, src, key_cols, target, scd_type=2, seq_col="last_modified_date"):
    """Render the AUTO CDC SQL template for one table and write it out."""
    # columns handled by the template itself
    special = {"hashkey", "data_version", "add_date", "delete_date",
               "last_modified_date", "is_active", "is_incremental"}
    # the template selects hashkey and data_version itself, so keep them (and row_hash) out of {cols}
    value_cols = [c for c in df.columns
                  if c not in key_cols and c not in special and c != "row_hash"]
    with open(TEMPLATE_PATH) as f:
        template = f.read()

    sql = template.format(
        src=src,                                   # upstream table, e.g. cat.schema.fact_claims
        target=target,                             # short name, e.g. fact_claims
        cols=", ".join(f"`{c}`" for c in [*key_cols, *value_cols]),
        keys=", ".join(f"`{c}`" for c in key_cols),
        seq_col=seq_col,
        scd_type=scd_type,
        track="TRACK HISTORY ON hashkey" if scd_type == 2 else "",
    )
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = f"{OUT_DIR}/ingestion_{target}.sql"
    with open(out_path, "w") as f:
        f.write(sql)
    return out_path

In [0]:
def add_audit_columns(df, keep_row_hash=False):
    "audit columns for tables that are not written through SCD_UPSERT_logic (facts, plain dimensions)"
    hk = col("row_hash") if "row_hash" in df.columns else lit(None).cast("string")
    out = df.withColumn("hashkey", hk)
    if not keep_row_hash:
        out = out.drop("row_hash")        # hashkey replaces it
    return (out
            .withColumn("add_date", lit(RUN_TS).cast("timestamp"))
            .withColumn("delete_date", to_timestamp(lit("2099-12-31 00:00:00")))
            .withColumn("last_modified_date", lit(RUN_TS).cast("timestamp"))
            .withColumn("is_active", lit(1))
            .withColumn("is_incremental", lit(1)))


def ensure_columns(df, target):
    "target exists from an earlier run: add the columns it is missing (new audit / config columns)"
    if not spark.catalog.tableExists(target):
        return
    have = {c.lower() for c in spark.table(target).columns}
    # row_hash is always replaced by hashkey; never add it to the target table
    missing = [f for f in df.schema.fields
               if f.name.lower() not in have and f.name.lower() != "row_hash"]
    if missing:
        cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in missing)
        spark.sql(f"ALTER TABLE {target} ADD COLUMNS ({cols})")
        logger.warning(f"{target}: added {[f.name for f in missing]}; rows loaded earlier hold NULL there")


def write_stream_both(df, target, cdm_table, vol_path):
    """Single streaming pass: each batch is written to both the UC table and the
    volume Delta path, avoiding a second stream on the same source."""
    def batch_writer(batch_df, batch_id):
        batch_df.write.format("delta").mode("append").saveAsTable(target)
        batch_df.write.format("delta").mode("append").save(vol_path)
    (df.writeStream
       .option("checkpointLocation", f"{CKPT_BASE}/{cdm_table}")
       .outputMode("append")
       .trigger(availableNow=True)
       .foreachBatch(batch_writer)
       .start()
       .awaitTermination())

def create_current_view(target):
    """<target>_v: same table, is_incremental = 1 only for the rows of the latest run.
    Computed on read, so the big fact table is never updated."""
    spark.sql(f"""
        CREATE OR REPLACE VIEW {target}_v AS
        SELECT * EXCEPT (is_incremental),
               CASE WHEN add_date = (SELECT MAX(add_date) FROM {target}) THEN 1 ELSE 0 END AS is_incremental
        FROM {target}""")


def load_fact(df, target, cdm_table, keep_row_hash=False, layer="ingestion"):
    "audit columns first (every new row: is_active = 1, is_incremental = 1), then append only"
    out = add_audit_columns(df, keep_row_hash)
    short_name = target.split('.')[-1]
    vol_path = get_vol_path(layer, short_name)
    # If the table exists but is missing audit columns (or they are all NULL from an
    # older run before audit columns were added), drop it (and the checkpoint) so
    # the stream recreates the table with all columns populated.
    if spark.catalog.tableExists(target):
        cols = {f.name for f in spark.table(target).schema.fields}
        stale = "is_incremental" not in cols
        if not stale and "is_incremental" in cols:
            stale = spark.table(target).filter(col("is_incremental").isNotNull()).limit(1).count() == 0
        if stale:
            logger.warning(f"{target}: stale schema (is_incremental missing or all NULL), dropping for recreation")
            spark.sql(f"DROP TABLE IF EXISTS {target}")
            dbutils.fs.rm(f"{CKPT_BASE}/{cdm_table}", True)
    ensure_columns(out, target)
    if out.isStreaming:
        write_stream_both(out, target, cdm_table, vol_path)
        # If the volume copy has no data (checkpoint already advanced, no new batches),
        # seed it from the UC table so the volume always has the full dataset
        try:
            vol_has_data = spark.read.format("delta").load(vol_path).limit(1).count() > 0
        except Exception:
            vol_has_data = False
        if not vol_has_data and spark.catalog.tableExists(target):
            logger.info(f"{target}: seeding volume copy from UC table (no new stream batches)")
            spark.table(target).write.format("delta").mode("overwrite").save(vol_path)
    else:
        out.write.format("delta").mode("append").saveAsTable(target)
        out.write.format("delta").mode("append").save(vol_path)
    create_current_view(target)
    return out

In [0]:
def create_ingestion_table(config, src_table,cdm_table, layer="ingestion", src_layer="staging"):
    cfg, src = config[layer], config[src_layer]
    map_df = read_mapping(cfg["mappath"])

    df, keys, scd, target_types = build_ingestion_df(map_df, cdm_table, src["catalog"], src["schema"])

    # config column, added after the hash so a version bump never looks like a data change
    if "data_version" not in df.columns:
        df = df.withColumn("data_version", lit(get_data_version(cfg)))

    # dropDuplicates needs a watermark on streams, and an empty key list would collapse the table to 1 row
    if not df.isStreaming and keys:
        df = df.dropDuplicates(keys)      # one row per key

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cfg['catalog']}`.`{cfg['schema']}`")
    target = f"{cfg['catalog']}.{cfg['schema']}.ingestion_{cdm_table}"
    is_dim = any(t in ("dim", "dimension") for t in target_types)

    if scd and not keys:
        logger.warning(f"{cdm_table}: scd = Y but no PRIMARY key in the mapping, nothing to match on. "
                       f"Loading append-only with audit columns; mark the business key PRIMARY to get SCD.")
        scd = False

    if scd and is_dim:
        # full-load dimension: batch MERGE, runs in this notebook
        ensure_columns(df, target)
        SCD_UPSERT_logic(df, keys, target, layer=layer)
    elif scd:
        # incremental fact: land the rows WITH the audit columns (the template reads hashkey and
        # data_version from this landing table), then generate the pipeline SQL
        out = load_fact(df, target, cdm_table, layer=layer)
        # the 4th arg is the SHORT name the template uses to build src_<t>/scd_<t>/<t>
        sql_path = scd_cdc_logic(out, target, keys, cdm_table)
        logger.info(f"{cdm_table}: generated {sql_path}; run the pipeline to build {cdm_table}")
    elif is_dim:
        short_name = target.split('.')[-1]
        vol_path = get_vol_path(layer, short_name)
        dim_df = add_audit_columns(df)
        (dim_df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true").saveAsTable(target))
        (dim_df.write.format("delta").mode("overwrite")
           .option("overwriteSchema", "true").save(vol_path))
    else:
        # non-SCD fact: append with audit columns
        load_fact(df, target, cdm_table, layer=layer)

    logger.info(f"Processed {target}")
    return df

In [0]:
config_path = '/Volumes/dab/healthcare_datasets/healthcare_vol/config/dev_config.json'

dbutils.widgets.text("source", "sphub");  source = dbutils.widgets.get("source")
dbutils.widgets.text("run_id", "manual"); run_id = dbutils.widgets.get("run_id")

config = load_config(config_path)
LOG_TABLE = config["control"]["log_table"]
TGT_PREFIX = "ingestion_"

mapping_rows = (read_mapping(config["ingestion"]["mappath"])
          .filter(col("is_active_map").cast("int") == 1)
          .groupBy(trim("cdm_table").alias("cdm_table"), trim("src_table").alias("src_table"))
          .agg(max(lower(trim("target_type"))).alias("target_type"))
          .collect())

known    = {r["cdm_table"].lower() for r in mapping_rows}
run_list = get_run_list(config["control"]["path"], "ingestion", source, known)
enabled  = {r["table_name"] for r in run_list}

tables = [r for r in mapping_rows if r["cdm_table"].lower() in enabled]
tables = sorted(tables, key=lambda r: 0 if r["target_type"] in ("dim", "dimension") else 1)

failed = []
for r in tables:
    t0 = datetime.now()
    try:
        create_ingestion_table(config, r["src_table"], r["cdm_table"])
        log_run(run_id, "ingestion", r["cdm_table"], "SUCCESS", t0)
    except Exception as e:
        logger.error(f"{r['cdm_table']} failed: {e}")
        log_run(run_id, "ingestion", r["cdm_table"], "FAILED", t0, str(e))
        failed.append(r["cdm_table"])

if failed:
    raise RuntimeError(f"Failed tables: {failed}")

2026-10-27 08:35:49,775 | INFO | __main__ | devproduct.dev.ingestion_provider: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/ingestion/ingestion_provider
2026-10-27 08:35:49,777 | INFO | __main__ | Processed devproduct.dev.ingestion_provider
2026-10-27 08:36:01,745 | INFO | __main__ | devproduct.dev.ingestion_patient: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/ingestion/ingestion_patient
2026-10-27 08:36:01,746 | INFO | __main__ | Processed devproduct.dev.ingestion_patient
2026-10-27 08:36:13,729 | INFO | __main__ | devproduct.dev.ingestion_procedure_code: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/healthcare_vol/data/ingestion/ingestion_procedure_code
2026-10-27 08:36:13,731 | INFO | __main__ | Processed devproduct.dev.ingestion_procedure_code
2026-10-27 08:36:25,400 | INFO | __main__ | devproduct.dev.ingestion_diagnosis_code: MERGE done, volume synced to /Volumes/dab/healthcare_datasets/heal